# Uchambuzi wa Dai la Gharama

Daftari hili linaonyesha jinsi ya kuunda mawakala ambao hutumia viunganishi kushughulikia gharama za safari kutoka kwa picha za risiti za ndani, kuunda barua pepe ya dai la gharama, na kuonyesha data za gharama kwa kutumia chati ya pai. Wakala huchagua kazi kwa nguvu kwa msingi wa muktadha wa kazi.

Hatua:
1. Wakala wa OCR hushughulikia picha ya risiti ya ndani na kutoa data za gharama za safari.
2. Wakala wa Barua pepe huunda barua pepe ya dai la gharama.

### Mfano wa hali ya gharama za safari:
Fikiria wewe ni mfanyakazi anaye wasafiri kwa mkutano wa biashara katika jiji jingine. Kampuni yako ina sera ya kurudisha gharama zote za safari zinazofaa. Hapa kuna muhtasari wa gharama zinazoweza kusababishwa na safari:
- Usafiri:
Tiketi za ndege kwenda na kurudi kutoka jiji lako la makazi hadi jiji la lengo.
Teksi au huduma za kuendesha magari kwa kwenda na kutoka uwanja wa ndege.
Usafiri wa ndani katika jiji la lengo (kama usafiri wa umma, magari ya kukodi, au teksi).

- Malazi:
Kubaki hoteli kwa usiku tatu katika hoteli ya biashara ya kiwango cha kati karibu na eneo la mkutano.

- Mlo:
Kikomo cha kila siku cha chakula kwa kifungua kinywa, chakula cha mchana, na chakula cha jioni, kulingana na sera ya kampuni ya malipo ya kila siku.

- Gharama Mbalimbali:
Ada za parkingi katika uwanja wa ndege.
Zamala za kupata intaneti katika hoteli.
Tips au ada ndogo za huduma.

- Nyaraka:
Unawasilisha risiti zote (tiketi za ndege, teksi, hoteli, chakula, n.k.) na ripoti kamili ya gharama kwa ajili ya malipo.


## Leta maktaba zinazohitajika

Leta maktaba na moduli zinazohitajika kwa ajili ya daftari hii.


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import base64
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

 ## Define Modeli za Matumizi

 Tengeneza نموذج ya Pydantic kwa matumizi binafsi na darasa la ExpenseFormatter kubadilisha swali la mtumiaji kuwa data ya matumizi iliyo na muundo.

 Kila matumizi yatawakilishwa kwa muundo:
 `{'date': '07-Mar-2025', 'description': 'flight to destination', 'amount': 675.99, 'category': 'Transportation'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## Kuelezesha Vifaa - Kutengeneza Barua Pepe

Unda kazi ya kifaa kutengeneza barua pepe ya kuwasilisha dai la matumizi.
- Kifaa hiki kinatumia kiungo cha `@tool` kutoka kwa Microsoft Agent Framework.
- Inahesabu jumla ya kiasi cha matumizi na kuandaa maelezo kuwa mwili wa barua pepe.


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# Zana ya Kuchukua Gharama za Safari Kutoka kwenye Picha za Risiti

Tengeneza kazi ya zana kuchukua gharama za safari kutoka kwenye picha za risiti.
- Zana hii inatumia `@tool` mtindo wa Microsoft Agent Framework.
- Inasoma picha ya risiti, kuibadilisha kuwa base64, na kurudisha data URI kwa ajili ya wakala kuchambua.


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## Kuchakata Matumizi

Tambua mawakala na waya wasiwe katika mtiririko mfululizo wa kazi kwa kutumia `WorkflowBuilder`.
- Wakala wa OCR hutokeza data ya matumizi iliyopangwa kutoka kwenye picha ya risiti kwa kutumia zana ya `load_receipt_image`.
- Wakala wa Barua Pepe hutumia data iliyotolewa na kutengeneza barua ya madai ya matumizi ya kibenki kwa kutumia zana ya `generate_expense_email`.
- `WorkflowBuilder` kwa `add_edge` huunda mfululizo wa kazi: Wakala wa OCR → Wakala wa Barua Pepe.


In [ ]:
ocr_agent = await provider.create_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = await provider.create_agent(
    name="EmailAgent",
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## Kazi kuu

Jenga mtiririko wa kazi wa mfululizo na uufanye utumie kushughulikia picha ya risiti na kuunda barua pepe ya dai la gharama.


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Tangazo la Hukumu**:
Nyaraka hii imetafsiriwa kwa kutumia huduma ya utafsiri wa AI [Co-op Translator](https://github.com/Azure/co-op-translator). Ingawa tunajitahidi kufanikisha usahihi, tafadhali fahamu kwamba tafsiri za kiotomatiki zinaweza kuwa na makosa au kupotosha. Nyaraka ya asili katika lugha yake ya asili inapaswa kuchukuliwa kama chanzo cha mamlaka. Kwa habari muhimu, tafsiri ya kitaalamu ya binadamu inashauriwa. Hatubebii dhima kwa kutoelewana au tafsiri potofu zitokanazo na matumizi ya tafsiri hii.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
